# Visualize `.npy` patches

Loads patches from `data/patches_128/` and displays them at full resolution.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "tests" else Path.cwd()
PATCH_DIR = REPO_ROOT / "data" / "patches_128_from_zip"
print("Patch dir:", PATCH_DIR, "exists =", PATCH_DIR.exists())

In [ ]:
files = sorted(PATCH_DIR.glob("*.npy"))
print(f"Found {len(files)} .npy files")
print("First 5:", [f.name for f in files[:5]])

sample = np.load(files[0])
print("Sample shape:", sample.shape, "dtype:", sample.dtype,
      "min/max:", float(sample.min()), float(sample.max()))

## Show a single patch at full resolution

Change `PATCH_NAME` to view any patch by filename (e.g. `img0040_r05_c10.npy`).

In [ ]:
PATCH_NAME = files[0].name  # or set to e.g. "img0040_r05_c10.npy"

arr = np.load(PATCH_DIR / PATCH_NAME)
if arr.ndim == 3 and arr.shape[0] in (1, 3):
    arr_disp = np.transpose(arr, (1, 2, 0)).squeeze()
else:
    arr_disp = arr.squeeze()

h, w = arr_disp.shape[:2]
dpi = 100
fig = plt.figure(figsize=(w / dpi, h / dpi), dpi=dpi)
ax = plt.Axes(fig, [0, 0, 1, 1])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(arr_disp, cmap="gray", interpolation="nearest")
plt.title(PATCH_NAME)
plt.show()
print("Displayed at", arr_disp.shape, "pixels (1:1).")

## Grid of N random patches (each shown at full resolution)

In [ ]:
N = 16          # number of patches
COLS = 4        # columns in the grid
SEED = 0

rng = np.random.default_rng(SEED)
picked = rng.choice(len(files), size=min(N, len(files)), replace=False)
picked_files = [files[i] for i in picked]

rows = (len(picked_files) + COLS - 1) // COLS
sample0 = np.load(picked_files[0]).squeeze()
h, w = sample0.shape[-2], sample0.shape[-1]
dpi = 100

fig, axes = plt.subplots(rows, COLS,
                         figsize=(COLS * w / dpi, rows * h / dpi),
                         dpi=dpi)
axes = np.atleast_2d(axes).ravel()
for ax, fp in zip(axes, picked_files):
    a = np.load(fp).squeeze()
    if a.ndim == 3 and a.shape[0] in (1, 3):
        a = np.transpose(a, (1, 2, 0)).squeeze()
    ax.imshow(a, cmap="gray", interpolation="nearest")
    ax.set_title(fp.name, fontsize=7)
    ax.set_axis_off()
for ax in axes[len(picked_files):]:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Reassemble one full image from its tiles

Filenames follow `imgXXXX_rRR_cCC.npy`. The cell below stitches all tiles of a chosen image into the full picture.

In [ ]:
import re

IMG_ID = "img0040"   # change to any imgXXXX present in PATCH_DIR

pat = re.compile(rf"^{IMG_ID}_r(\d+)_c(\d+)\.npy$")
tiles = {}
for fp in PATCH_DIR.glob(f"{IMG_ID}_*.npy"):
    m = pat.match(fp.name)
    if m:
        tiles[(int(m.group(1)), int(m.group(2)))] = fp

if not tiles:
    raise FileNotFoundError(f"No tiles for {IMG_ID} in {PATCH_DIR}")

rows = max(r for r, _ in tiles) + 1
cols = max(c for _, c in tiles) + 1
sample = np.load(next(iter(tiles.values()))).squeeze()
ph, pw = sample.shape[-2], sample.shape[-1]

full = np.zeros((rows * ph, cols * pw), dtype=sample.dtype)
for (r, c), fp in tiles.items():
    a = np.load(fp).squeeze()
    full[r * ph:(r + 1) * ph, c * pw:(c + 1) * pw] = a

H, W = full.shape
dpi = 100
fig = plt.figure(figsize=(W / dpi, H / dpi), dpi=dpi)
ax = plt.Axes(fig, [0, 0, 1, 1])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(full, cmap="gray", interpolation="nearest")
plt.title(f"{IMG_ID}  reassembled  {rows}x{cols} tiles  ->  {H}x{W} px")
plt.show()